# 04_als_model

In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession

# 1. Point to your Hadoop bin folder
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

# 2. Pin Python
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Start Spark with the RawLocalFileSystem workaround
spark = SparkSession.builder \
    .appName("Spotify_ALS") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .getOrCreate()

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS

# Path to the train / validation / test split produced in notebook 02_2
SPLIT_DIR = "../outputs/train_test_split"

users_train = spark.read.csv(f"{SPLIT_DIR}/users_train",      header=True, inferSchema=True)
users_val   = spark.read.csv(f"{SPLIT_DIR}/users_validation", header=True, inferSchema=True)
users_test  = spark.read.csv(f"{SPLIT_DIR}/users_test",       header=True, inferSchema=True)

print(f"Training rows:   {users_train.count():,}")
print(f"Validation rows: {users_val.count():,}")
print(f"Test rows:       {users_test.count():,}")
users_train.show(5)

Training rows:   9,107,237
Validation rows: 1,136,551
Test rows:       1,140,062
+-------+-------+-----------+--------------------+--------------------+-------------------+
|user_id|song_id|interaction|    user_id_original|          track_name|        artist_name|
+-------+-------+-----------+--------------------+--------------------+-------------------+
|      0| 196963|        1.0|00055176fea33f6e0...|Another You (Anot...|Against The Current|
|      0| 920045|        1.0|00055176fea33f6e0...|       Gone Too Soon|        Simple Plan|
|      0|1143595|        1.0|00055176fea33f6e0...|   If You Don't Know|5 Seconds Of Summer|
|      0|1730322|        1.0|00055176fea33f6e0...| One Of Those Nights|       Shawn Mendes|
|      0|1946760|        1.0|00055176fea33f6e0...|         Risk It All|          The Vamps|
+-------+-------+-----------+--------------------+--------------------+-------------------+
only showing top 5 rows


In [3]:
from pyspark.ml.recommendation import ALS

# Initialize the ALS model using the columns your teammate already prepared
als = ALS(
    maxIter=10,                      
    regParam=0.1,                    
    userCol="user_id",         
    itemCol="song_id",        
    ratingCol="interaction",   
    coldStartStrategy="drop",        
    implicitPrefs=True               
)

print("Training the ALS Model... ")
model = als.fit(users_train)
print("Training Complete!")

Training the ALS Model... 
Training Complete!


In [4]:
# ============================================================
# VALIDATION — HR@K + MRR@K via negative sampling
# ============================================================
# For each held-out (user, song) pair we ask the model to rank the
# real song against 99 random "negative" songs the user didn't play.
# Out of those 100 candidates:
#   HR@K  = 1 if the real song lands in the top-K, else 0
#   MRR@K = 1/rank if it lands in the top-K, else 0
#           (rank-1 → 1.0, rank-2 → 0.5, ..., rank-10 → 0.1)
# Then average across all evaluated pairs.
#
# A random model gets HR@10 ≈ 10/100 = 0.10. A good model: 0.4–0.7.
#
# Robustness: we collect the FULL user/item factor tables once per
# model (no joins, no isin filters with 50K literals — both choke on
# Windows after a few training jobs). Lookup happens in Python dicts.
# Costs ~30 s per eval but is bullet-proof.

import numpy as np
from pyspark.sql import functions as F

K           = 10
EVAL_PAIRS  = 5000
N_NEGATIVES = 99
SEED        = 42

users_train.cache()

# Pull the song catalog once. Skip rebuild if it's already there.
if "ITEM_CATALOG" not in globals():
    print("Building song catalog for negative sampling...")
    ITEM_CATALOG = users_train.select("song_id").distinct().toPandas()["song_id"].to_numpy()
    N_CATALOG    = len(ITEM_CATALOG)
    print(f"Catalog: {N_CATALOG:,} unique songs")
else:
    print(f"Re-using cached catalog ({N_CATALOG:,} songs)")


def sample_eval_pairs(test_df, seed=SEED):
    """Pick EVAL_PAIRS random (user_id, positive_song_id) pairs from test_df."""
    total = test_df.count()
    frac = min(1.0, EVAL_PAIRS * 3.0 / total)
    rows = (
        test_df.select("user_id", "song_id")
        .sample(False, frac, seed=seed)
        .limit(EVAL_PAIRS)
        .collect()
    )
    return [(r.user_id, r.song_id) for r in rows]


def _build_candidates(pairs, seed=SEED):
    """For each (user, positive) -> [positive, neg1, ..., neg99]  (positive at idx 0)."""
    rng = np.random.default_rng(seed)
    out = []
    for user_id, pos in pairs:
        idx = rng.choice(N_CATALOG, size=N_NEGATIVES + 5, replace=False)
        sampled = ITEM_CATALOG[idx]
        sampled = sampled[sampled != pos][:N_NEGATIVES]
        out.append((user_id, np.concatenate(([pos], sampled))))
    return out


def _collect_factors(model, candidates=None):
    """Collect FULL user/item factor tables. Bullet-proof: no joins,
    no isin with 50K literals, no Python workers. ~15 MB + 480 MB."""
    u_pd = model.userFactors.toPandas()
    i_pd = model.itemFactors.toPandas()
    u_vec = {row["id"]: np.asarray(row["features"], dtype=np.float32) for _, row in u_pd.iterrows()}
    i_vec = {row["id"]: np.asarray(row["features"], dtype=np.float32) for _, row in i_pd.iterrows()}
    return u_vec, i_vec


def _rank_of_positive(scores):
    """Rank (1-indexed) of the positive — always at index 0. Stable sort handles ties."""
    order = np.argsort(-scores, kind="stable")
    return 1 + int(np.where(order == 0)[0][0])


def _aggregate_hr_mrr(score_iter, k):
    hr = mrr = n = 0
    for scores in score_iter:
        if scores is None:
            continue
        rank = _rank_of_positive(scores)
        n += 1
        if rank <= k:
            hr  += 1
            mrr += 1.0 / rank
    if n == 0:
        return {"hr@k": 0.0, "mrr@k": 0.0, "n": 0}
    return {"hr@k": hr / n, "mrr@k": mrr / n, "n": n}


def evaluate_model(model, test_df, k=K):
    pairs        = sample_eval_pairs(test_df)
    candidates   = _build_candidates(pairs)
    u_vec, i_vec = _collect_factors(model)

    def score_each():
        for user_id, items in candidates:
            u = u_vec.get(user_id)
            if u is None:                       # cold-start user → skip
                yield None
                continue
            scores = np.empty(len(items), dtype=np.float32)
            for idx, sid in enumerate(items):
                v = i_vec.get(int(sid))
                scores[idx] = u @ v if v is not None else -np.inf
            yield scores

    return _aggregate_hr_mrr(score_each(), k)


def print_metrics(label, m, k=K):
    print(f"{label:<24}  HR@{k}={m['hr@k']:.4f}   MRR@{k}={m['mrr@k']:.4f}   (n={m['n']})")


print(f"\nEvaluating baseline ALS on {EVAL_PAIRS} test pairs...")
baseline_metrics = evaluate_model(model, users_test)
print_metrics("Baseline ALS", baseline_metrics)

Building song catalog for negative sampling...
Catalog: 2,434,622 unique songs

Evaluating baseline ALS on 5000 test pairs...
Baseline ALS              HR@10=0.6387   MRR@10=0.4301   (n=4999)


In [5]:
# ============================================================
# Hyperparameter tuning
# ============================================================
# Small grid over the parameters that matter most for implicit ALS:
#   rank      — latent factor dimension (model capacity)
#   regParam  — L2 regularisation (overfitting control)
#   alpha     — implicit confidence multiplier (Hu, Koren, Volinsky 2008);
#               larger alpha = more weight on observed plays vs unobserved
#
# Each candidate is scored on the dedicated validation split with
# the same HR@K + MRR@K negative-sampling protocol. Selection
# criterion: HR@K. The test set stays sealed until the final cell.

users_train.cache()
users_val.cache()

grid = [
    {"rank": 10, "regParam": 0.1,  "alpha": 1.0},   # baseline-ish defaults
    {"rank": 50, "regParam": 0.1,  "alpha": 1.0},   # higher capacity
    {"rank": 50, "regParam": 0.1,  "alpha": 20.0},  # higher confidence weight
]

results = []
for params in grid:
    print(f"\n>> Training {params}")
    candidate = ALS(
        maxIter=10,
        userCol="user_id",
        itemCol="song_id",
        ratingCol="interaction",
        coldStartStrategy="drop",
        implicitPrefs=True,
        **params,
    ).fit(users_train)
    m = evaluate_model(candidate, users_val, k=K)
    print(f"   HR@{K}={m['hr@k']:.4f}   MRR@{K}={m['mrr@k']:.4f}")
    results.append((params, m))

best_params, best_metrics = max(results, key=lambda r: r[1]["hr@k"])
print(f"\nBest config (by HR@{K}): {best_params}")
print_metrics("  validation", best_metrics)


>> Training {'rank': 10, 'regParam': 0.1, 'alpha': 1.0}
   HR@10=0.6322   MRR@10=0.4200

>> Training {'rank': 50, 'regParam': 0.1, 'alpha': 1.0}
   HR@10=0.6742   MRR@10=0.4797

>> Training {'rank': 50, 'regParam': 0.1, 'alpha': 20.0}
   HR@10=0.7008   MRR@10=0.5195

Best config (by HR@10): {'rank': 50, 'regParam': 0.1, 'alpha': 20.0}
  validation              HR@10=0.7008   MRR@10=0.5195   (n=5000)


In [6]:
# ============================================================
# Final model — retrain on full training data with best params
# ============================================================
print(f"Training final ALS with best params: {best_params}")
final_als = ALS(
    maxIter=15,                       # a few more iterations for the final fit
    userCol="user_id",
    itemCol="song_id",
    ratingCol="interaction",
    coldStartStrategy="drop",
    implicitPrefs=True,
    **best_params,
)
final_model = final_als.fit(users_train)
print("Training complete.")

print("\nEvaluating final model on TEST set...")
final_metrics = evaluate_model(final_model, users_test, k=K)
print_metrics("Tuned ALS (test)", final_metrics)

# # Side-by-side comparison
# print(f"\nComparison @ K={K}  (random ≈ 0.10 HR, 0.029 MRR)")
# print(f"{'metric':<8}{'popularity':>14}{'baseline ALS':>16}{'tuned ALS':>14}")
# print("-" * 52)
# for name in ["hr@k", "mrr@k"]:
#     print(f"{name:<8}{popularity_metrics[name]:>14.4f}{baseline_metrics[name]:>16.4f}{final_metrics[name]:>14.4f}")

Training final ALS with best params: {'rank': 50, 'regParam': 0.1, 'alpha': 20.0}
Training complete.

Evaluating final model on TEST set...


Py4JJavaError: An error occurred while calling o531.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 7 in stage 2220.0 failed 1 times, most recent failure: Lost task 7.0 in stage 2220.0 (TID 2899) (DESKTOP-OHUB31N executor driver): TaskResultLost (result lost from block manager)
Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$3(DAGScheduler.scala:3122)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:3122)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:3114)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:3114)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGScheduler.scala:1303)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1$adapted(DA

In [14]:
from pyspark.sql.functions import explode, col
import os
os.makedirs(spark.sparkContext._temp_dir, exist_ok=True)
# 1. INPUT: pick a user
user_id = 15
user_to_test = spark.createDataFrame([(user_id,)], ["user_id"])

# 2. Over-fetch from the tuned model so we still have TARGET_K left
#    after dropping songs the user already heard in training.
TARGET_K = 10
OVER_FETCH = 50
raw_recs = final_model.recommendForUserSubset(user_to_test, OVER_FETCH)

exploded_recs = raw_recs.select(
    "user_id",
    explode("recommendations").alias("rec"),
).select(
    "user_id",
    col("rec.song_id").alias("song_id"),
    col("rec.rating").alias("confidence_score"),
)

# 3. FILTER: drop songs the user already listened to in training —
#    we want NEW recommendations, not playback prediction.
already_heard = users_train.filter(col("user_id") == user_id).select("song_id").distinct()
fresh_recs = exploded_recs.join(already_heard, "song_id", "left_anti")

# 4. TRANSLATE IDs -> human-readable names, keep the top TARGET_K
song_dictionary = users_train.select("song_id", "track_name", "artist_name").distinct()
final_playlist = (
    fresh_recs.join(song_dictionary, "song_id")
    .orderBy(col("confidence_score").desc())
    .limit(TARGET_K)
)

print(f"🎧 Top {TARGET_K} NEW recommendations for user {user_id} (tuned ALS) 🎧")
final_playlist.select("track_name", "artist_name", "confidence_score").show(TARGET_K, truncate=False)

🎧 Top 10 NEW recommendations for user 15 (tuned ALS) 🎧
+----------------------------------------------+------------------+----------------+
|track_name                                    |artist_name       |confidence_score|
+----------------------------------------------+------------------+----------------+
|Let's Stay Together                           |Al Green          |0.43738422      |
|September                                     |Earth, Wind & Fire|0.41989735      |
|Son Of A Preacher Man                         |Dusty Springfield |0.41870788      |
|(Sittin' On) The Dock Of The Bay              |Otis Redding      |0.40927875      |
|Superstition                                  |Stevie Wonder     |0.40901554      |
|Ain't No Mountain High Enough - Stereo Version|Marvin Gaye       |0.40350446      |
|Go Your Own Way                               |Fleetwood Mac     |0.4014381       |
|Sir Duke                                      |Stevie Wonder     |0.39388347      |
|My Girl  

In [15]:
from pyspark.sql.functions import col

print("🎧 What User 20 ACTUALLY Listened To (Training Data) 🎧")

# Filter the training data for only User 15, sort by what they played the most
actual_history = users_train.filter(col("user_id") == 15) \
                            .orderBy(col("interaction").desc())

# Show their top 15 most played songs (including how they are stored)
actual_history.select(
    "user_id", 
    "user_id_original", 
    "track_name", 
    "artist_name", 
    "interaction"
).show(15, truncate=False)

🎧 What User 20 ACTUALLY Listened To (Training Data) 🎧
+-------+--------------------------------+------------------------------------------------------+----------------------------+-----------+
|user_id|user_id_original                |track_name                                            |artist_name                 |interaction|
+-------+--------------------------------+------------------------------------------------------+----------------------------+-----------+
|15     |002e678084db2e201d2721bf5af4e54c|Do You Wanna Dance? - 2001 - Remaster;Mono            |The Beach Boys              |2.0        |
|15     |002e678084db2e201d2721bf5af4e54c|Happy Xmas (War Is Over)                              |John Lennon                 |1.0        |
|15     |002e678084db2e201d2721bf5af4e54c|Basic Instructions Before Leaving Earth               |GZA                         |1.0        |
|15     |002e678084db2e201d2721bf5af4e54c|In the Back of My Mind - 2001 - Remaster;Mono         |The Beach Boys 

In [ ]:
# ============================================================
# Save the trained model so we don't have to retrain next time.
# ============================================================
import os

MODEL_DIR = "../outputs/models"
MODEL_PATH = f"{MODEL_DIR}/als_final"
os.makedirs(MODEL_DIR, exist_ok=True)

final_model.write().overwrite().save(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

# To reload later:
#   from pyspark.ml.recommendation import ALSModel
#   final_model = ALSModel.load(MODEL_PATH)